# C04 技术指标策略与分散化（在线版）

使用真实在线行情对比 Bollinger 与 RSI 策略。


In [ ]:
START_DATE = "2022-01-01"
END_DATE = "2024-12-31"
UNIVERSE = ["510300.XSHG", "510500.XSHG", "159915.XSHE", "518880.XSHG"]

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import rqdatac

# 方式1（推荐课堂演示）：把教育版 license 直接粘贴到 PASSWD
PASSWD = "在这里粘贴你的教育版license"

# 方式2（不想明文写在 notebook）：注释掉上面一行，改用环境变量
# import os
# PASSWD = os.getenv("RQDATAC_LICENSE", "")

if PASSWD and ("在这里粘贴" not in PASSWD):
    rqdatac.init('license', PASSWD)
    print("rqdatac 初始化成功")
else:
    print("请先填写 PASSWD，再运行本单元")


In [ ]:
close = rqdatac.get_price(
    UNIVERSE,
    start_date=START_DATE,
    end_date=END_DATE,
    fields="close",
    adjust_type="pre",
    expect_df=False,
)
close = close.droplevel(0, axis=1) if isinstance(close.columns, pd.MultiIndex) else close
ret = close.pct_change().fillna(0)
close.head()


## Bollinger 主策略


In [ ]:
def boll_pos(series, n=20, entry=-1.0, exit_level=0.0):
    ma = series.rolling(n).mean()
    sd = series.rolling(n).std()
    z = (series - ma) / (sd + 1e-12)

    out = pd.Series(0.0, index=series.index)
    holding = 0.0
    for i in range(len(series)):
        if np.isnan(z.iloc[i]):
            out.iloc[i] = holding
            continue
        if holding == 0 and z.iloc[i] < entry:
            holding = 1.0
        elif holding == 1 and z.iloc[i] > exit_level:
            holding = 0.0
        out.iloc[i] = holding
    return out

boll_position = pd.DataFrame({c: boll_pos(close[c]) for c in close.columns})
boll_ret = (boll_position.shift(1).fillna(0) * ret).mean(axis=1)
boll_nav = (1 + boll_ret).cumprod()


## RSI 对照策略


In [ ]:
def rsi(series, n=14):
    delta = series.diff()
    up = delta.clip(lower=0).rolling(n).mean()
    down = (-delta.clip(upper=0)).rolling(n).mean()
    rs = up / (down + 1e-12)
    return 100 - (100 / (1 + rs))


def rsi_pos(series, n=14, low=30, high=55):
    rr = rsi(series, n)
    out = pd.Series(0.0, index=series.index)
    holding = 0.0
    for i in range(len(series)):
        if np.isnan(rr.iloc[i]):
            out.iloc[i] = holding
            continue
        if holding == 0 and rr.iloc[i] < low:
            holding = 1.0
        elif holding == 1 and rr.iloc[i] > high:
            holding = 0.0
        out.iloc[i] = holding
    return out

rsi_position = pd.DataFrame({c: rsi_pos(close[c]) for c in close.columns})
rsi_ret = (rsi_position.shift(1).fillna(0) * ret).mean(axis=1)
rsi_nav = (1 + rsi_ret).cumprod()


In [ ]:
perf = pd.DataFrame({"Bollinger": boll_nav, "RSI": rsi_nav})
fig, ax = plt.subplots(figsize=(10, 4))
perf.plot(ax=ax, title="C04 在线版策略净值对比")
ax.set_ylabel("NAV")
plt.show()


def metrics(r):
    ann_ret = r.mean() * 252
    ann_vol = r.std() * np.sqrt(252)
    sharpe = ann_ret / (ann_vol + 1e-12)
    mdd = ((1 + r).cumprod() / (1 + r).cumprod().cummax() - 1).min()
    return ann_ret, ann_vol, sharpe, mdd

summary = pd.DataFrame(
    [metrics(boll_ret), metrics(rsi_ret)],
    columns=["annual_return", "annual_vol", "sharpe", "max_drawdown"],
    index=["Bollinger", "RSI"],
).round(4)
summary
